# CSE428 Project — Pet Segmentation & Breed Classification

**Dataset:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) · 37 breeds · 3,680 trainval / 3,666 test images

**Models:** U-Net and Attention U-Net with a breed-classifier head on the shared encoder (trained jointly)

**Contents:** data exploration → base U-Net → Attention U-Net → results (mIoU, Dice, pixel accuracy · accuracy, precision, recall, F1 on train/val/test)

> Training runs in **resumable checkpoint chunks**: each notebook version saves its state to the output, and the next run (or a groupmate) resumes from it — never start from scratch.

## 0. Repository sync — code comes from GitHub, no copy-paste

All project code lives in a public repo. This cell clones it (fresh session) or pulls the latest version (existing session).

In [ ]:
import os, sys

REPO_URL = "https://github.com/shahriar-abid/cse428-pets.git"
REPO_DIR = "/kaggle/working/cse428-pets"

# Data lives OUTSIDE the captured repo snapshot so it is not bundled into the
# notebook output (keeps output small/fast and only the checkpoints persist).
os.environ.setdefault("CSE428_DATA_ROOT", "/kaggle/working/data")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
os.environ["CSE428_DEPLOY_DIR"] = os.getcwd()
sys.path.insert(0, REPO_DIR)

# Restore checkpoints attached from a previous version.
# Kaggle presents attached datasets EITHER as a .zip OR already-extracted
# (auto-extracted) folder. Handle both so outputs/<model>/checkpoints/best.pth
# always exists for resume, comparison and instant eval.
if os.path.isdir("/kaggle/input"):
    import glob, shutil, tempfile, os
    import torch as _torch
    def _restore_one(best_path):
        meta = _torch.load(best_path, map_location="cpu", weights_only=False)
        mname = meta.get("cfg", {}).get("model", {}).get("name", "unet")
        target = os.path.join(REPO_DIR, "outputs", mname, "checkpoints")
        os.makedirs(target, exist_ok=True)
        src = os.path.dirname(best_path)
        for f in ("best.pth", "last.pth"):
            sp = os.path.join(src, f)
            if os.path.exists(sp):
                shutil.copy(sp, os.path.join(target, f))
        mdir = os.path.dirname(src)  # .../outputs/<model> root (contains history/results)
        for j in ("history.json", "results.json"):
            sp = os.path.join(mdir, j)
            if os.path.exists(sp):
                shutil.copy(sp, os.path.join(os.path.dirname(target), j))
        print(f"restored model -> outputs/{mname} from {best_path}")

    # 1) already-extracted folders
    for best_path in sorted(glob.glob(os.path.join("/kaggle/input", "**", "checkpoints", "best.pth"), recursive=True)):
        try:
            _restore_one(best_path)
        except Exception as e:
            print("skip restore (no valid ckpt):", best_path, e)

    # 2) zipped archives (unpack then same logic)
    for arch in sorted(glob.glob(os.path.join("/kaggle/input", "**", "*_artifacts.zip"), recursive=True)):
        tmp = tempfile.mkdtemp()
        try:
            shutil.unpack_archive(arch, tmp)
            for best_path in glob.glob(os.path.join(tmp, "**", "checkpoints", "best.pth"), recursive=True):
                _restore_one(best_path)
        except Exception as e:
            print("skip archive (unreadable):", arch, e)
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
print("repo ready:", os.getcwd())


## 1. Setup

In [ ]:
import yaml
import torch

from src.utils import seed_everything, get_device, check_device, resolve_output_dir

CFG = yaml.safe_load(open("configs/config.yaml"))
seed_everything(CFG["seed"])
DEVICE = check_device(get_device())
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(DEVICE))
print("device:", DEVICE)
print("config:", CFG)

## 2. Dataset & Exploration

**Oxford-IIIT Pet** — 3,680 trainval + 3,666 test images across 37 breeds.

Each image ships with a **trimap**: `1` = foreground (pet), `2` = background, `3` = boundary (not classified). Per the project guidelines, boundary pixels are merged into the foreground, giving a binary mask: `background → 0`, `foreground + boundary → 1`.

Split: 90% train / 10% validation from trainval (deterministic, seeded — identical across sessions so checkpointed training resumes on the same data). The official test set is used for testing.

In [ ]:
from src.data import get_loaders

DATASETS, LOADERS = get_loaders(
    root=CFG["data"]["root"],
    img_size=CFG["data"]["img_size"],
    val_frac=CFG["data"]["val_frac"],
    seed=CFG["seed"],
    augment=CFG["data"]["augment"],
    batch_size=CFG["data"]["batch_size"],
    num_workers=CFG["data"]["num_workers"],
    download=True,
)
for name, d in DATASETS.items():
    print(f"{name}: {len(d)} samples ({len(d.classes)} classes)")

### 2.1 Images with mask overlays (3x3, required format)

In [ ]:
import matplotlib.pyplot as plt
from src.viz import plot_overlay_grid

fig = plot_overlay_grid(DATASETS["val"], figsize=(12, 12))
plt.show()

## 3. Base U-Net — segmentation + classification (trained jointly)

**Architecture** (Ronneberger et al., 2015):
- **Segmentation head** on the decoder: 1×1 conv → binary mask, trained with BCE + Dice loss
- **Classifier head** on the encoder bottleneck: global average pooling → linear → 37 breeds, cross-entropy
- **Joint loss**: `L = L_seg + λ · L_cls` (both heads train together, sharing the encoder)

**Checkpoint-chunked training:** each run trains *from the last completed epoch* up to `train.epochs_total`, saving `checkpoints/last.pth` (full state) and `checkpoints/best.pth` (best val mIoU) to the notebook output.

**To continue training later (or as a groupmate):** *Add Input → this notebook's previous version output* — the trainer finds the checkpoint automatically. Just bump `epochs_total` and run.

In [ ]:
from src.models import build_model
from src.train import Trainer, find_resume_checkpoint

MODEL_NAME = "unet"                  # "unet" | "attention_unet"
CFG["model"]["name"] = MODEL_NAME
CFG["train"]["epochs_total"] = 80    # target: 60 epochs total (resumes where left off)

OUT_DIR = os.path.join(resolve_output_dir(CFG["output"]["dir"]), MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
print("output dir:", OUT_DIR)
model = build_model(CFG).to(DEVICE)
resume_ckpt = find_resume_checkpoint(out_dir=OUT_DIR, model_name=MODEL_NAME)
print("resume checkpoint:", resume_ckpt)
trainer = Trainer(model, LOADERS, DEVICE, CFG, out_dir=OUT_DIR, resume=resume_ckpt)

In [ ]:
history = trainer.fit()

### 3.1 Training curves — loss + metric update per epoch (requirement)

In [ ]:
from src.viz import plot_history

fig = plot_history(trainer.history)
plt.show()

### 3.2 Result summary — train / validation / test (requirement)

Segmentation: **mIoU, Dice coefficient, pixel accuracy** · Classification: **accuracy, precision, recall, F1** (macro-averaged over the 37 breeds).

In [ ]:
import pandas as pd

results = trainer.final_report()
rows = [{"split": split, **m["seg"], **m["cls"]} for split, m in results.items()]
pd.DataFrame(rows).set_index("split").round(4)

### 3.2b Persisted outputs (for resume / groupmate handoff)

Verify the checkpoint + report files are on disk inside `/kaggle/working` so Kaggle captures them into this version's output — the next chunk (or a groupmate) attaches this version and resumes from `best.pth`/`last.pth`.

In [ ]:
_od, _name = OUT_DIR, f"{MODEL_NAME}_artifacts"
import glob, os, shutil

_files = sorted(glob.glob(os.path.join(_od, "**", "*.*"), recursive=True))
for p in _files:
    print(f"{os.path.getsize(p):>12,} bytes  {p}")
_ckpt = glob.glob(os.path.join(_od, "checkpoints", "best.pth"))
if _ckpt:
    _art = shutil.make_archive(_name, "zip", _od)
    print("handoff archive:", _art, os.path.getsize(_art), "bytes")
else:
    print(f"WARNING: no best.pth under {_od}; skipped creating {_name}.zip "
          f"(nothing trained/persisted for this model yet).")



### 3.3 Predictions — image | ground truth | model output (required format)

Random validation samples; titles show the predicted breed with confidence and the per-image foreground IoU.

In [ ]:
from src.viz import plot_prediction_grid

fig = plot_prediction_grid(DATASETS["val"], trainer.model, DEVICE, nrows=3, figsize=(13, 13))
plt.show()

## 4. Attention U-Net — segmentation + classification

*(Oktay et al., 2018)* — additive attention gates re-weight each skip connection
using the coarser decoder feature as the gating signal. Same classifier head on the
bottleneck, trained jointly. Identical protocol to section 3, with its own output dir
so the two models never overwrite each other.


In [ ]:
# Attention U-Net: separate trainer + output dir so it does not clash with the base U-Net.
ATTN_MODEL = "attention_unet"
CFG["model"]["name"] = ATTN_MODEL
ATTN_OUT = os.path.join(resolve_output_dir(CFG["output"]["dir"]), ATTN_MODEL)
os.makedirs(ATTN_OUT, exist_ok=True)
print("output dir:", ATTN_OUT)

attn_model = build_model(CFG).to(DEVICE)
attn_resume = find_resume_checkpoint(out_dir=ATTN_OUT, model_name=ATTN_MODEL)
print("attention resume checkpoint:", attn_resume)
attn_trainer = Trainer(attn_model, LOADERS, DEVICE, CFG, out_dir=ATTN_OUT, resume=attn_resume)


In [ ]:
attn_history = attn_trainer.fit()


In [ ]:
from src.viz import plot_history
fig = plot_history(attn_trainer.history)
plt.show()


In [ ]:
import pandas as pd
attn_results = attn_trainer.final_report()
attn_rows = [{"split": s, **m["seg"], **m["cls"]} for s, m in attn_results.items()]
pd.DataFrame(attn_rows).set_index("split").round(4)


In [ ]:
from src.viz import plot_prediction_grid
fig = plot_prediction_grid(DATASETS["val"], attn_trainer.model, DEVICE, nrows=3, figsize=(13, 13))
plt.show()


In [ ]:
_od, _name = ATTN_OUT, f"{ATTN_MODEL}_artifacts"
import glob, os, shutil

_files = sorted(glob.glob(os.path.join(_od, "**", "*.*"), recursive=True))
for p in _files:
    print(f"{os.path.getsize(p):>12,} bytes  {p}")
_ckpt = glob.glob(os.path.join(_od, "checkpoints", "best.pth"))
if _ckpt:
    _art = shutil.make_archive(_name, "zip", _od)
    print("handoff archive:", _art, os.path.getsize(_art), "bytes")
else:
    print(f"WARNING: no best.pth under {_od}; skipped creating {_name}.zip "
          f"(nothing trained/persisted for this model yet).")



## 5. U-Net vs Attention U-Net — comparison & discussion

Both models use identical data, split, augmentation and hyperparameters (base_channels=32,
same scheduler/loss). The only difference is the attention gates on the decoder. Below: a
side-by-side of final test metrics from each `best.pth`, plus total parameters.


In [ ]:
import pandas as pd
from src.models import build_model

def load_best(out_dir, name):
    import os, torch, glob
    ck = glob.glob(os.path.join(out_dir, "checkpoints", "best.pth"))
    if not ck:
        return None
    return torch.load(ck[0], map_location="cpu", weights_only=False)

rows = {}
for name, od in [("unet", OUT_DIR), ("attention_unet", ATTN_OUT)]:
    ck = load_best(od, name)
    if ck is None:
        rows[name] = {"status": "no best.pth yet", "parameters": None,
                      "best_val_mIoU": None, "best_epoch": None}
        continue
    n_params = sum(p.numel() for p in build_model(ck["cfg"]).parameters())
    rows[name] = {"status": "ok", "parameters": n_params,
                  "best_val_mIoU": round(ck.get("best_miou", -1), 4),
                  "best_epoch": ck.get("epoch")}

compare = pd.DataFrame(rows).T
compare["parameters"] = compare["parameters"].astype(str)
display(compare)

print("\nDiscussion:")
print("- Attention gates add attention parameters to each skip path; expect comparable or"
      "slightly better mIoU thanks to focused foreground features, at modest parameter cost.")
print("- Both heads share the same bottleneck classifier, so classification metrics track"
      "the segmentation backbone representational quality.")


## 6. Deployment — instant evaluation on a random image (saved weights)

The instructor will supply a new image at evaluation time. We must **not** retrain —
we restore the saved checkpoint and run one forward pass. `best.pth` already embeds
the model config + class labels, so inference needs only torch.


In [ ]:
import subprocess, glob, os, torch

# Instant evaluation on a random image with a SAVED checkpoint (no training).
# Resolve which model to show: set EVAL_MODEL to "unet" or "attention_unet".
# Defaults to whichever model has a best.pth on disk (the one just trained).
EVAL_MODEL = "attention_unet"   # "unet" | "attention_unet"
EVAL_DIR = {"unet": OUT_DIR, "attention_unet": ATTN_OUT}[EVAL_MODEL]
best = sorted(glob.glob(os.path.join(EVAL_DIR, "checkpoints", "best.pth")))
print(f"checkpoint [{EVAL_MODEL}]:", best[0] if best else "MISSING")
assert best, f"No best.pth for {EVAL_MODEL} at {EVAL_DIR} - train it first."

# Grab one real test image to prove instant inference (no training).
from src.data import resolve_data_root, PetSegDataset
root = resolve_data_root(CFG["data"]["root"])
ds = PetSegDataset(root, split="test", img_size=CFG["data"]["img_size"])
idx = int(torch.randint(len(ds), (1,)))
img_pil, (label, _) = ds.base[int(idx)]
img_path = os.path.join(EVAL_DIR, "_random_eval_input.png")
img_pil.save(img_path)
print("saved random image ->", img_path)

out_png = os.path.join(EVAL_DIR, "instant_eval_overlay.png")
r = subprocess.run(
    ["python", "scripts/predict.py", "--checkpoint", best[0],
     "--image", img_path, "--out", out_png],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
from IPython.display import Image as DispImg
if os.path.exists(out_png):
    display(DispImg(out_png))


## 7. Bonus Task 1 — Classifier backbone comparison

Compare 3 well-established classifier architectures (ResNet-18, MobileNetV3-Small,
EfficientNet-B0) for the breed-classification head, using the same splits/data and the
same Trainer. Each is trained for a short equal budget so results are directly comparable.


In [ ]:
import pandas as pd
from src.models import build_backbone_classifier
from src.train import Trainer
import torch

BONUS_EPOCHS = 10     # equal budget for a fair comparison
BACKBONES = ["resnet18", "mobilenet_v3_small", "efficientnet_b0"]
BONUS_CFG = dict(CFG)                       # shallow copy
BONUS_CFG["data"] = dict(CFG["data"])
BONUS_CFG["train"] = dict(CFG["train"])
BONUS_CFG["model"] = dict(CFG["model"])
BONUS_CFG["train"]["epochs_total"] = BONUS_EPOCHS

bonus_rows = []
for bb in BACKBONES:
    print(f"\n=== training backbone: {bb} ===")
    clf = build_backbone_classifier(bb).to(DEVICE)
    tr = Trainer(clf, LOADERS, DEVICE, BONUS_CFG, out_dir=os.path.join(OUT_DIR, "bonus", bb))
    tr.fit()
    results = tr.final_report()
    test_cls = results["test"]["cls"]
    bonus_rows.append({"backbone": bb, **test_cls})

bonus_df = pd.DataFrame(bonus_rows).set_index("backbone")
display(bonus_df.round(4))
print("\nBest backbone by test accuracy:", bonus_df["acc"].idxmax())


### Bonus 1 — Interpretation
- **ResNet-18:** strong general-purpose backbone, larger capacity.
- **MobileNetV3-Small:** lightweight/efficient, trades a bit of accuracy for speed.
- **EfficientNet-B0:** compound-scaled, good accuracy-per-parameter.
We report test accuracy, macro precision/recall/F1 for each on the same held-out test set.